# Multi-Model Trading Architecture Training

This notebook trains the three-model ensemble for Solana memecoin trading:

1. **Model 1 (Screener)**: XGBoost classifier - determines if a token is worth monitoring
2. **Model 2 (Entry)**: LSTM/Hybrid - determines optimal entry timing
3. **Model 3 (Exit)**: XGBoost + Rules - determines optimal exit point

## Architecture Overview

```
Token Stream --> [SCREENER] --> [ENTRY] --> [EXIT] --> Trade Execution
                 (XGBoost)     (LSTM)      (Rules+ML)
```

## Requirements
- GPU recommended (A100, L4, T4)
- ~1-2 hours total training time

## 1. Setup

In [ ]:
# GitHub Configuration
GITHUB_USERNAME = "tr4m0ryp"  # Your GitHub username
GITHUB_TOKEN = "YOUR_GITHUB_TOKEN"  # Your Personal Access Token
GITHUB_REPO_URL = "https://github.com/tr4m0ryp/GMGN_TradingBot.git"

print("GitHub credentials configured")
print(f"Username: {GITHUB_USERNAME}")
print(f"Repository: {GITHUB_REPO_URL}")

In [ ]:
import os
import subprocess

os.chdir('/content')

repo_name = GITHUB_REPO_URL.split('/')[-1].replace('.git', '')
repo_path = f'/content/{repo_name}'

if os.path.exists(repo_path):
    print(f"Repository '{repo_name}' already exists at {repo_path}")
    print("Pulling latest changes...")
    os.chdir(repo_path)
    subprocess.run(['git', 'pull'], capture_output=True, text=True)
else:
    clone_url = GITHUB_REPO_URL.replace('https://', f'https://{GITHUB_USERNAME}:{GITHUB_TOKEN}@')
    result = subprocess.run(['git', 'clone', clone_url], capture_output=True, text=True, check=True)
    print(f"Successfully cloned repository to {repo_path}")

import sys
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

ai_model_v2_path = f'{repo_path}/ai_model_v2/src'
if ai_model_v2_path not in sys.path:
    sys.path.insert(0, ai_model_v2_path)

os.chdir(f'{repo_path}/ai_model_v2')
print(f"Current directory: {os.getcwd()}")

In [ ]:
# Install dependencies
!pip install -q torch numpy pandas xgboost scikit-learn matplotlib tqdm

import torch
import numpy as np
import xgboost as xgb

print(f"[OK] PyTorch {torch.__version__}")
print(f"[OK] XGBoost {xgb.__version__}")
print(f"[OK] NumPy {np.__version__}")

In [ ]:
# Check GPU
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    total_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
    device = 'cuda'
    torch.backends.cudnn.benchmark = True
    print(f"[GPU] {gpu_name} | {total_mem:.1f}GB | CUDA {torch.version.cuda}")
else:
    device = 'cpu'
    print("[WARN] No GPU available - training will be slower")

print(f"\nDevice: {device}")

## 2. Import Modules

In [ ]:
# Import project modules
from config import (
    ScreenerConfig,
    EntryConfig,
    ExitConfig,
    BacktestConfig,
    get_gpu_config,
)
from data import (
    load_raw_data,
    split_tokens_chronological,
    compute_dataset_statistics,
)
from data.dataset import (
    prepare_screener_data,
    prepare_entry_data,
    prepare_exit_data,
)
from models import (
    ScreenerModel,
    EntryModel,
    ExitModel,
    TradingPipeline,
    create_entry_model,
)
from training import (
    train_screener_model,
    train_entry_model,
    train_exit_model,
)
from backtesting import (
    run_backtest,
    print_metrics_report,
)
from utils import set_seed, get_device

print("[OK] All modules imported successfully!")
print("\nAvailable components:")
print("  - ScreenerModel: XGBoost entry worthiness classifier")
print("  - EntryModel: LSTM entry timing optimizer")
print("  - ExitModel: Exit point optimizer with risk rules")
print("  - TradingPipeline: Complete trading system")

## 3. Configure Training

In [ ]:
# Get GPU-optimized config
gpu_config = get_gpu_config()
print(f"GPU Config: {gpu_config}")

# Data paths
repo_name = GITHUB_REPO_URL.split('/')[-1].replace('.git', '')
DATA_PATH = f'/content/{repo_name}/ai_data/data/combined_tokens.csv'
OUTPUT_DIR = f'/content/{repo_name}/ai_model_v2/models/checkpoints'

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Set random seed
set_seed(42)

print(f"\nData path: {DATA_PATH}")
print(f"Output dir: {OUTPUT_DIR}")

In [ ]:
# Create model configs with GPU-optimized settings
screener_config = ScreenerConfig(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    scale_pos_weight=4.0,  # Handle class imbalance
)

entry_config = EntryConfig(
    input_dim=14,
    hidden_dim=gpu_config['hidden'],
    num_layers=gpu_config['lstm_layers'],
    batch_size=gpu_config['batch'],
    total_epochs=50,
    patience=10,
)

exit_config = ExitConfig(
    n_estimators=100,
    max_depth=5,
    stop_loss_pct=0.25,
    trailing_stop_pct=0.15,
    profit_target_pct=2.00,
)

print("Configuration created:")
print(f"  Screener: {screener_config.n_estimators} trees, depth {screener_config.max_depth}")
print(f"  Entry: hidden={entry_config.hidden_dim}, layers={entry_config.num_layers}, batch={entry_config.batch_size}")
print(f"  Exit: stop_loss={exit_config.stop_loss_pct:.0%}, trailing={exit_config.trailing_stop_pct:.0%}")

## 4. Load and Analyze Data

In [ ]:
# Load raw data
print("Loading data...")
tokens = load_raw_data(DATA_PATH)

# Compute statistics
stats = compute_dataset_statistics(tokens)

print(f"\n{'='*60}")
print("DATASET STATISTICS")
print(f"{'='*60}")
print(f"Total tokens: {stats['num_tokens']}")
print(f"\nLifespan (seconds):")
print(f"  Mean: {stats['lifespan']['mean']:.1f}")
print(f"  Median: {stats['lifespan']['median']:.1f}")
print(f"\nPeak ratio:")
print(f"  Median: {stats['peak_ratio']['median']:.2f}x")
print(f"  75th percentile: {stats['peak_ratio']['percentile_75']:.2f}x")
print(f"  90th percentile: {stats['peak_ratio']['percentile_90']:.2f}x")
print(f"\nSuccess rates:")
print(f"  2x: {stats['success_rates']['2x']:.1%}")
print(f"  4x: {stats['success_rates']['4x']:.1%}")
print(f"  10x: {stats['success_rates']['10x']:.1%}")
print(f"\nDeath reasons:")
for reason, count in stats['death_reasons'].items():
    print(f"  {reason}: {count}")

In [ ]:
# Split data chronologically
train_tokens, val_tokens, test_tokens = split_tokens_chronological(
    tokens, train_ratio=0.70, val_ratio=0.15, test_ratio=0.15
)

print(f"\nData split:")
print(f"  Train: {len(train_tokens)} tokens")
print(f"  Val: {len(val_tokens)} tokens")
print(f"  Test: {len(test_tokens)} tokens")

## 5. Train Model 1: Screener (XGBoost)

In [ ]:
print("Training Model 1: Entry Worthiness Screener...\n")

screener_model, screener_results = train_screener_model(
    data_path=DATA_PATH,
    output_dir=OUTPUT_DIR,
    config=screener_config,
    decision_time=30,
    verbose=1,
)

print(f"\nScreener training complete!")
print(f"Test ROC-AUC: {screener_results['eval_results']['metrics']['roc_auc']:.4f}")

## 6. Train Model 2: Entry Timing (LSTM)

In [ ]:
print("Training Model 2: Entry Timing Optimizer...\n")

entry_model, entry_results = train_entry_model(
    data_path=DATA_PATH,
    output_dir=OUTPUT_DIR,
    config=entry_config,
    device=device,
    verbose=1,
)

print(f"\nEntry model training complete!")
print(f"Test accuracy: {entry_results['test_metrics']['accuracy']:.4f}")

## 7. Train Model 3: Exit Optimizer (XGBoost + Rules)

In [ ]:
print("Training Model 3: Exit Point Optimizer...\n")

exit_model, exit_results = train_exit_model(
    data_path=DATA_PATH,
    output_dir=OUTPUT_DIR,
    config=exit_config,
    verbose=1,
)

print(f"\nExit model training complete!")
print(f"Test accuracy: {exit_results['test_metrics']['accuracy']:.4f}")

## 8. Backtesting

In [ ]:
# Create trading pipeline
pipeline = TradingPipeline(
    screener_model=screener_model,
    entry_model=entry_model,
    exit_model=exit_model,
)

print("Trading pipeline created!")

In [ ]:
# Run backtest on test tokens
print("Running backtest on test set...\n")

backtest_result = run_backtest(
    tokens=test_tokens,
    pipeline=pipeline,
    verbose=1,
)

In [ ]:
# Print detailed metrics
print_metrics_report(backtest_result)

## 9. Save Models

In [ ]:
# Save complete pipeline
pipeline.save(OUTPUT_DIR)
print(f"Pipeline saved to: {OUTPUT_DIR}")

# List saved files
!ls -la {OUTPUT_DIR}

In [ ]:
# Save to Google Drive (if available)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    
    import shutil
    DRIVE_PATH = '/content/drive/MyDrive/gmgn_models/multi_model'
    os.makedirs(DRIVE_PATH, exist_ok=True)
    
    # Copy model files
    for fname in os.listdir(OUTPUT_DIR):
        src = os.path.join(OUTPUT_DIR, fname)
        dst = os.path.join(DRIVE_PATH, fname)
        shutil.copy(src, dst)
        print(f"[SAVED] {fname}")
    
    print(f"\nModels saved to Google Drive: {DRIVE_PATH}")
except Exception as e:
    print(f"Google Drive not available: {e}")
    print(f"Models saved locally at: {OUTPUT_DIR}")

## 10. Training Summary

In [ ]:
print("\n" + "="*70)
print("                   TRAINING COMPLETE")
print("="*70)

print(f"\n[MODEL 1] Screener (XGBoost)")
print(f"  ROC-AUC: {screener_results['eval_results']['metrics']['roc_auc']:.4f}")
print(f"  Precision: {screener_results['eval_results']['metrics']['precision']:.4f}")
print(f"  Recall: {screener_results['eval_results']['metrics']['recall']:.4f}")

print(f"\n[MODEL 2] Entry Timing (LSTM)")
print(f"  Best epoch: {entry_results['best_epoch']}")
print(f"  Test accuracy: {entry_results['test_metrics']['accuracy']:.4f}")

print(f"\n[MODEL 3] Exit Optimizer (XGBoost + Rules)")
print(f"  Test accuracy: {exit_results['test_metrics']['accuracy']:.4f}")

print(f"\n[BACKTEST] Test Set Performance")
print(f"  Win rate: {backtest_result.win_rate:.1%}")
print(f"  Profit factor: {backtest_result.profit_factor:.2f}")
print(f"  Net P&L: {backtest_result.net_pnl:.4f} SOL")
print(f"  Max drawdown: {backtest_result.max_drawdown:.2%}")
print(f"  Sharpe ratio: {backtest_result.sharpe_ratio:.2f}")

print(f"\n[OUTPUT] {OUTPUT_DIR}")
print("="*70)